In [0]:
# Databricks widgets for portable configuration
dbutils.widgets.text("catalog", "workspace", "Catalog Name")
dbutils.widgets.text("schema_bronze", "cbl_bronze", "Bronze Schema")
dbutils.widgets.text("schema_silver", "cbl_silver", "Silver Schema")
dbutils.widgets.text("schema_gold", "cbl_gold", "Gold Schema")

# Read widget values into variables
catalog = dbutils.widgets.get("catalog")
schema_bronze = dbutils.widgets.get("schema_bronze")
schema_silver = dbutils.widgets.get("schema_silver")
schema_gold = dbutils.widgets.get("schema_gold")

# Print configuration
print("Gold Layer Configuration:")
print(f"  Catalog: {catalog}")
print(f"  Bronze Schema: {schema_bronze}")
print(f"  Silver Schema: {schema_silver}")
print(f"  Gold Schema: {schema_gold}")
print(f"\nGold Tables:")
print(f"  dim_outlet: {catalog}.{schema_gold}.dim_outlet")
print(f"  dim_product: {catalog}.{schema_gold}.dim_product")
print(f"  dim_distributor: {catalog}.{schema_gold}.dim_distributor")
print(f"  fact_sales_daily: {catalog}.{schema_gold}.fact_sales_daily")

In [0]:
from pyspark.sql.functions import col, row_number, when, trim, upper, current_timestamp, lit
from pyspark.sql.window import Window
import uuid

# Generate batch ID
batch_id = str(uuid.uuid4())

# Source and target
source_table = f"{catalog}.{schema_silver}.kna1_customer"
target_table = f"{catalog}.{schema_gold}.dim_outlet"

print(f"Building gold outlet dimension: {target_table}")
print(f"Source: {source_table}")
print(f"Batch ID: {batch_id}\n")

# Read silver customer table
df_customer = spark.table(source_table)

print(f"Silver customer rows: {df_customer.count():,}\n")

# Step 1: Filter out deleted customers (LOEVM = 'X')
df_active = df_customer.filter((col("deletion_flag") != "X") | col("deletion_flag").isNull())

print(f"After excluding deleted (LOEVM='X'): {df_active.count():,}\n")

# Step 2: Deduplicate by customer_id, keeping most recent record
window_spec = Window.partitionBy("customer_id").orderBy(col("created_date").desc(), col("_ingest_timestamp").desc())

df_deduped = (df_active
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

print(f"After deduplication: {df_deduped.count():,}\n")

# Step 3: Rename and select columns for dim_outlet
df_outlet = (df_deduped
    .select(
        col("customer_id").alias("outlet_id"),
        col("customer_name").alias("outlet_name"),
        col("street_address").alias("address"),
        col("city"),
        col("region"),
        col("account_group").alias("zone"),
        col("customer_tier").alias("tier"),
        col("area_type").alias("location_type"),
        col("created_date").alias("onboarded_date"),
        col("credit_limit"),
        col("cooler_count").alias("active_promos_count"),
        col("is_exclusive").alias("loyalty_status"),
        col("latitude"),
        col("longitude"),
        col("is_active")
    )
    # Add gold layer metadata
    .withColumn("_transform_timestamp", current_timestamp())
    .withColumn("_batch_id", lit(batch_id))
)

print("Sample outlet dimension data:")
display(df_outlet.limit(10))

# Write to gold with overwrite
df_outlet.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

final_count = df_outlet.count()

print(f"\n✓ Gold outlet dimension created successfully!")
print(f"  Table: {target_table}")
print(f"  Rows written: {final_count:,}")

In [0]:
from pyspark.sql.functions import col, sum as _sum, count, countDistinct, current_timestamp, lit
import uuid

# Generate batch ID
batch_id = str(uuid.uuid4())

# Source and target
source_table = f"{catalog}.{schema_silver}.fact_billing"
target_table = f"{catalog}.{schema_gold}.fact_sales_daily"

print(f"Building gold sales fact: {target_table}")
print(f"Source: {source_table}")
print(f"Batch ID: {batch_id}\n")

# Read silver fact_billing
df_billing = spark.table(source_table)

print(f"Silver fact_billing rows: {df_billing.count():,}\n")

# Aggregate to daily grain: billing_date x customer_id x material_id
df_sales_daily = (df_billing
    .groupBy("billing_date", "customer_id", "material_id")
    .agg(
        _sum("quantity").alias("total_units"),
        _sum("net_value").alias("total_net_value"),
        countDistinct("billing_doc").alias("transaction_count")
    )
    # Add gold layer metadata
    .withColumn("_transform_timestamp", current_timestamp())
    .withColumn("_batch_id", lit(batch_id))
)

print("Sample aggregated data:")
display(df_sales_daily.limit(10))

# Write to gold with overwrite
df_sales_daily.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

final_count = df_sales_daily.count()

print(f"\n✓ Gold sales fact created successfully!")
print(f"  Table: {target_table}")
print(f"  Rows written: {final_count:,}")

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit
import uuid

# Generate batch ID
batch_id = str(uuid.uuid4())

# Source and target
source_table = f"{catalog}.{schema_silver}.dim_product"
target_table = f"{catalog}.{schema_gold}.dim_product"

print(f"Building gold product dimension: {target_table}")
print(f"Source: {source_table}")
print(f"Batch ID: {batch_id}\n")

# Read silver dim_product
df_product = spark.table(source_table)

print(f"Silver product rows: {df_product.count():,}\n")

# Add gold layer metadata
df_gold_product = (df_product
    .withColumn("_transform_timestamp", current_timestamp())
    .withColumn("_batch_id", lit(batch_id))
)

print("Sample product dimension data:")
display(df_gold_product.limit(10))

# Write to gold with overwrite
df_gold_product.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

final_count = df_gold_product.count()

print(f"\n✓ Gold product dimension created successfully!")
print(f"  Table: {target_table}")
print(f"  Rows written: {final_count:,}")

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit
import uuid

# Generate batch ID
batch_id = str(uuid.uuid4())

# Source and target
source_table = f"{catalog}.{schema_silver}.dim_distributor"
target_table = f"{catalog}.{schema_gold}.dim_distributor"

print(f"Building gold distributor dimension: {target_table}")
print(f"Source: {source_table}")
print(f"Batch ID: {batch_id}\n")

# Read silver dim_distributor
df_distributor = spark.table(source_table)

print(f"Silver distributor rows: {df_distributor.count():,}\n")

# Add gold layer metadata
df_gold_distributor = (df_distributor
    .withColumn("_transform_timestamp", current_timestamp())
    .withColumn("_batch_id", lit(batch_id))
)

print("Sample distributor dimension data:")
display(df_gold_distributor.limit(10))

# Write to gold with overwrite
df_gold_distributor.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

final_count = df_gold_distributor.count()

print(f"\n✓ Gold distributor dimension created successfully!")
print(f"  Table: {target_table}")
print(f"  Rows written: {final_count:,}")

In [0]:
# Summary metrics for gold fact_sales_daily
query = f"""
SELECT 
  COUNT(*) as row_count,
  SUM(total_units) as total_units,
  SUM(total_net_value) as total_net_value
FROM {catalog}.{schema_gold}.fact_sales_daily
"""

result = spark.sql(query)
display(result)

# Store row count for assertion in next cell
metrics = result.collect()[0]
row_count = metrics['row_count']
total_units = metrics['total_units']
total_net_value = metrics['total_net_value']

print(f"\nMetrics Summary:")
print(f"  Row count: {row_count:,}")
print(f"  Total units: {total_units:,.3f}")
print(f"  Total net value: ${total_net_value:,.2f}")

In [0]:
# CRITICAL VALIDATION: Row count must match expected value
EXPECTED_ROW_COUNT = 241050

query = f"""
SELECT COUNT(*) as row_count
FROM {catalog}.{schema_gold}.fact_sales_daily
"""

actual_row_count = spark.sql(query).collect()[0]['row_count']

if actual_row_count != EXPECTED_ROW_COUNT:
    error_msg = (
        f"ASSERTION FAILED: Gold layer row count mismatch!\n"
        f"  Expected: {EXPECTED_ROW_COUNT:,} rows\n"
        f"  Actual:   {actual_row_count:,} rows\n"
        f"  Difference: {actual_row_count - EXPECTED_ROW_COUNT:+,} rows\n\n"
        f"This indicates a data quality issue. Investigation required before proceeding."
    )
    raise AssertionError(error_msg)

print(f"✓ ASSERTION PASSED: Row count is exactly {actual_row_count:,} as expected")